In [9]:
import os
import time

import cv2
import pandas as pd
import torch
from natsort import natsorted
from ultralytics import YOLO

from code_programm.path import get_path_weight_model

In [10]:
print(torch.cuda.device_count())
print(torch.cuda.get_device_name())
model_speed = YOLO(get_path_weight_model('speed_recognition.pt'))

1
NVIDIA GeForce GTX 1080 Ti


In [11]:
wheel_ets_train_speed = pd.DataFrame(columns=['speed'])

In [12]:
paths = [
    r'D:\Dataset_for_autopilot\2024-04-02 19-11-55',
    r'D:\Dataset_for_autopilot\2024-04-02 19-09-18',
    r'D:\Dataset_for_autopilot\2024-04-02 19-07-31',
    r'D:\Dataset_for_autopilot\2024-04-02 19-05-23',
    r'D:\Dataset_for_autopilot\2024-04-02 19-01-56',
    r'D:\Dataset_for_autopilot\2024-04-02 19-00-41',
    r'D:\Dataset_for_autopilot\2024-04-02 18-58-46',
    r'D:\Dataset_for_autopilot\2024-04-02 18-57-24',
    r'D:\Dataset_for_autopilot\2024-04-02 18-55-25',
]

In [13]:
for j in paths:
    path_i = os.path.join(j, f'speed')
    if os.path.exists(f'{path_i}') and os.path.isdir(f'{path_i}'):
        speed_files = [os.path.join(path_i, file) for file in os.listdir(path_i) if file.endswith('.png')]
        print("Полные пути к файлам в папке:")
        speed_files = natsorted(speed_files)
        print(speed_files[0])
    else:
        print("Указанный путь не существует или не является папкой.")
    length = len(wheel_ets_train_speed)
    start_time = time.time()
    for file in speed_files:
        bgra_image = cv2.imread(file, cv2.IMREAD_UNCHANGED)
        scaled_image = cv2.cvtColor(bgra_image, cv2.COLOR_BGRA2BGR)
        # scaled_image = cv2.resize(scaled_image, None, fx=2, fy=2, interpolation=cv2.INTER_LINEAR)
        results = model_speed.predict(scaled_image, conf=0.8, device='cuda', verbose=False, show=False)
        sorted_objects = sorted(
            ({'class': int(cls), 'confidence': float(conf), 'xmin': int(xmin), 'ymin': int(ymin), 'xmax': int(xmax),
              'ymax': int(ymax)}
             for result in results for obj in result.boxes.data for xmin, ymin, xmax, ymax, conf, cls in
             (obj.tolist(),)),
            key=lambda obj: obj['xmin']
        )
        if sorted_objects:
            speed = ''.join(str(obj['class']) for obj in sorted_objects)
        else:
            speed = wheel_ets_train_speed.iloc[len(wheel_ets_train_speed) - 1]
        wheel_ets_train_speed.loc[len(wheel_ets_train_speed)] = int(speed)
    print('Изображений:', len(wheel_ets_train_speed) - length, '\nСек:', time.time() - start_time, '\nCек\изображение:',
          (time.time() - start_time) / (len(wheel_ets_train_speed) - length), '\n')
print('Всего:', len(wheel_ets_train_speed))

Полные пути к файлам в папке:
D:\Dataset_for_autopilot\2024-04-02 19-11-55\speed\2024-04-02 19-11-55_0.png
Изображений: 231 
Сек: 5.593702554702759 
Cек\изображение: 0.024215162574470817 

Полные пути к файлам в папке:
D:\Dataset_for_autopilot\2024-04-02 19-09-18\speed\2024-04-02 19-09-18_0.png
Изображений: 591 
Сек: 15.7321138381958 
Cек\изображение: 0.026619481959722167 

Полные пути к файлам в папке:
D:\Dataset_for_autopilot\2024-04-02 19-07-31\speed\2024-04-02 19-07-31_0.png
Изображений: 1817 
Сек: 52.804611682891846 
Cек\изображение: 0.029061426352719782 

Полные пути к файлам в папке:
D:\Dataset_for_autopilot\2024-04-02 19-05-23\speed\2024-04-02 19-05-23_0.png
Изображений: 1436 
Сек: 36.75816226005554 
Cек\изображение: 0.025597606030679346 

Полные пути к файлам в папке:
D:\Dataset_for_autopilot\2024-04-02 19-01-56\speed\2024-04-02 19-01-56_0.png
Изображений: 3705 
Сек: 58.112348794937134 
Cек\изображение: 0.015684844479065353 

Полные пути к файлам в папке:
D:\Dataset_for_autopi

In [14]:
wheel_ets_train_speed = wheel_ets_train_speed.drop(wheel_ets_train_speed.tail(1).index, axis = 0)
wheel_ets_train_speed.info()

<class 'pandas.core.frame.DataFrame'>
Index: 12781 entries, 0 to 12780
Data columns (total 1 columns):
 #   Column  Non-Null Count  Dtype
---  ------  --------------  -----
 0   speed   12781 non-null  int64
dtypes: int64(1)
memory usage: 199.7 KB


In [15]:
wheel_ets_train_speed.to_csv(r'C:\PycharmProjects\ETS_Autopilot\dataset_for_wheel_nn\X_train_speed_rnn.csv', index=False)